# F03A — SERAPHIM-A : L'Architecte
> *"Avant de couper, dessine. Chaque milliseconde est une décision de guerre."*
> — Ordre de la Rose Sacrée, Adepta Sororitas

```
╔══════════════════════════════════════════════════════════════╗
║   FRÉGATE F03A — SERAPHIM-A : L'ARCHITECTE                  ║
║   Rôle    : Analyse BPM + Détection sections + JSON directives║
║   IN      : trend_music.mp3  (F03_SERAPHIM/IN/)              ║
║   OUT     : directives.json  (F03_SERAPHIM/CODEBASE/)        ║
║   Stack   : Librosa · Matplotlib · Gradio headless           ║
╚══════════════════════════════════════════════════════════════╝
```

---

## Ordre des Cellules

| # | Cellule | Description |
|---|---------|-------------|
| 1 | INIT | Monter Drive, cloner SANCTORUM, définir chemins |
| 2 | INSTALLATION | Installer librosa, gradio, matplotlib |
| 3 | INTERFACE | Gradio — L'Architecte (BPM + éditeur directives) |
| 4 | SR_CUSTOS | Vérification directives.json + check-in flotte |

---
## CELLULE 1 — INIT

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELLULE 1 — INIT                                       ║
# ╚══════════════════════════════════════════════════════════╝

import os, sys, json
from google.colab import drive

if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

DRIVE_ROOT      = '/content/drive/MyDrive/SANCTORUM'
F03_IN          = f'{DRIVE_ROOT}/F03_SERAPHIM/IN'
F03_OUT         = f'{DRIVE_ROOT}/F03_SERAPHIM/OUT'
F03_CODEBASE    = f'{DRIVE_ROOT}/F03_SERAPHIM/CODEBASE'
F03_TRACKING    = f'{DRIVE_ROOT}/F03_SERAPHIM/TRACKING'
LIBER_DRIVE     = f'{DRIVE_ROOT}/liber_sanctorum.json'
SANCTORUM_DIR   = '/content/SANCTORUM'

if not os.path.exists(SANCTORUM_DIR):
    print('[INIT] Clonage SANCTORUM...')
    !git clone https://github.com/kioka8877-ux/SANCTORUM.git {SANCTORUM_DIR} -q
else:
    !git -C {SANCTORUM_DIR} pull -q

if SANCTORUM_DIR not in sys.path:
    sys.path.insert(0, SANCTORUM_DIR)

for d in [F03_IN, F03_OUT, F03_CODEBASE, F03_TRACKING]:
    os.makedirs(d, exist_ok=True)

fleet_status = 'unknown'
if os.path.exists(LIBER_DRIVE):
    with open(LIBER_DRIVE) as f:
        liber = json.load(f)
    fleet_status = liber.get('fleet_status', 'unknown')

# Lister les musiques disponibles dans IN/
AUDIO_EXTS = {'.mp3', '.wav', '.flac', '.ogg', '.m4a', '.aac'}
musics_in = [f for f in os.listdir(F03_IN)
             if os.path.splitext(f)[1].lower() in AUDIO_EXTS]

print(f'[INIT] fleet_status : {fleet_status}')
print(f'[INIT] Musiques IN  : {musics_in or "(aucune — uploader via Gradio)"}')
print('[INIT] Prêt.')

---
## CELLULE 2 — INSTALLATION

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELLULE 2 — INSTALLATION                               ║
# ╚══════════════════════════════════════════════════════════╝

print('[INSTALL] Dépendances F03A SERAPHIM-A...')
!pip install -q librosa>=0.10.0
!pip install -q soundfile pydub numpy scipy
!pip install -q matplotlib gradio>=4.31.0
!pip install -q pandas
!apt-get install -qq ffmpeg 2>/dev/null
print('[INSTALL] Terminé.')

---
## CELLULE 3 — INTERFACE GRADIO — L'ARCHITECTE

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELLULE 3 — INTERFACE GRADIO — L'ARCHITECTE            ║
# ║  BPM + Détection sections + Éditeur directives JSON      ║
# ╚══════════════════════════════════════════════════════════╝

import os, sys, json, shutil, time, tempfile
from datetime import datetime, timezone
import numpy as np
import soundfile as sf
import librosa
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd
import gradio as gr

DRIVE_ROOT    = '/content/drive/MyDrive/SANCTORUM'
F03_IN        = f'{DRIVE_ROOT}/F03_SERAPHIM/IN'
F03_CODEBASE  = f'{DRIVE_ROOT}/F03_SERAPHIM/CODEBASE'
F03_TRACKING  = f'{DRIVE_ROOT}/F03_SERAPHIM/TRACKING'
LIBER_DRIVE   = f'{DRIVE_ROOT}/liber_sanctorum.json'
SANCTORUM_DIR = '/content/SANCTORUM'
LOCAL_CODEBASE = f'{SANCTORUM_DIR}/F03_SERAPHIM/CODEBASE'

ROLES = ['queue', 'loop', 'tete', 'drop', 'bridge', 'outro']
SEG_COLUMNS = ['role', 'start', 'end', 'loops', 'speed', 'reverse',
                'volume_pct', 'fade_in_ms', 'fade_out_ms']


# ╔══════════════════════════════════════════════════════════╗
# ║  ANALYSE AUDIO                                           ║
# ╚══════════════════════════════════════════════════════════╝

def analyze_audio(audio_path: str) -> dict:
    """
    Analyse complète : BPM, durée, beats, sections structurelles.
    Retourne un dict prêt pour construire la timeline initiale.
    """
    y, sr = librosa.load(audio_path, sr=None, mono=True)
    duration = librosa.get_duration(y=y, sr=sr)

    # BPM + beats
    tempo, beat_frames = librosa.beat.beat_track(y=y, sr=sr)
    beat_times = librosa.frames_to_time(beat_frames, sr=sr).tolist()
    bpm = float(round(tempo, 2))

    # Détection de sections (MFCC + segmentation agglomérative)
    hop = 512
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=12, hop_length=hop)
    n_sections = min(6, max(3, int(duration // 15)))
    bounds_frames = librosa.segment.agglomerative(mfcc, k=n_sections)
    bounds_times  = librosa.frames_to_time(bounds_frames, sr=sr, hop_length=hop).tolist()
    bounds_times  = [0.0] + [round(t, 2) for t in bounds_times] + [round(duration, 2)]

    # Construire les segments initiaux
    role_map = ['queue', 'loop', 'loop', 'tete', 'drop', 'outro']
    segments = []
    for i in range(len(bounds_times) - 1):
        seg_dur = bounds_times[i+1] - bounds_times[i]
        role = role_map[min(i, len(role_map)-1)]
        segments.append({
            'role'        : role,
            'start'       : round(bounds_times[i], 2),
            'end'         : round(bounds_times[i+1], 2),
            'loops'       : 1 if role != 'loop' else max(1, int(30 / seg_dur)),
            'speed'       : 1.0,
            'reverse'     : False,
            'volume_pct'  : 80 if role == 'loop' else 100,
            'fade_in_ms'  : 0 if i == 0 else 15,
            'fade_out_ms' : 300 if i == len(bounds_times)-2 else 15,
        })

    return {
        'y': y, 'sr': sr,
        'bpm': bpm, 'duration': round(duration, 2),
        'beat_times': beat_times[:64],
        'bounds_times': bounds_times,
        'segments': segments,
    }


def plot_analysis(y, sr, bpm, beat_times, bounds_times, audio_path) -> str:
    """Waveform + grille BPM + sections détectées."""
    fig, ax = plt.subplots(figsize=(14, 4), facecolor='#0a0a0f')
    ax.set_facecolor('#111')

    # Waveform
    duration = librosa.get_duration(y=y, sr=sr)
    t = np.linspace(0, duration, len(y))
    ax.fill_between(t, y, alpha=0.5, color='#2c3e50')
    ax.plot(t, y, color='#3498db', linewidth=0.3, alpha=0.7)

    # Beat grid
    for bt in beat_times:
        ax.axvline(bt, color='#ffffff', alpha=0.08, linewidth=0.5)

    # Sections
    colors_sec = ['#c0392b', '#e67e22', '#f1c40f', '#2ecc71', '#1abc9c', '#9b59b6']
    for i in range(len(bounds_times) - 1):
        c = colors_sec[i % len(colors_sec)]
        ax.axvspan(bounds_times[i], bounds_times[i+1], alpha=0.12, color=c)
        ax.axvline(bounds_times[i], color=c, alpha=0.6, linewidth=1.2)
        mid = (bounds_times[i] + bounds_times[i+1]) / 2
        ax.text(mid, 0.85, f'S{i+1}', color=c, fontsize=8,
                ha='center', transform=ax.get_xaxis_transform(),
                fontfamily='monospace')

    ax.set_title(f'SERAPHIM-A — {os.path.basename(audio_path)} | BPM: {bpm}',
                 color='#3498db', fontfamily='monospace', fontsize=11)
    ax.set_xlabel('Temps (s)', color='#555', fontfamily='monospace')
    ax.tick_params(colors='#555')
    ax.spines[:].set_color('#333')

    plt.tight_layout()
    out = '/tmp/seraphim_a_analysis.png'
    plt.savefig(out, dpi=110, bbox_inches='tight', facecolor='#0a0a0f')
    plt.close(fig)
    return out


def segments_to_df(segments: list) -> pd.DataFrame:
    return pd.DataFrame(segments, columns=SEG_COLUMNS)


_last_analysis = {}


def run_analysis(use_drive, upload_file):
    global _last_analysis
    status = []

    # Résoudre l'entrée
    if use_drive:
        candidates = sorted(
            [os.path.join(F03_IN, f) for f in os.listdir(F03_IN)
             if os.path.splitext(f)[1].lower() in {'.mp3','.wav','.flac','.ogg','.m4a'}]
        )
        if not candidates:
            return None, None, None, '[ERREUR] Aucun fichier audio dans F03_SERAPHIM/IN/'
        audio_path = candidates[0]
    elif upload_file is not None:
        audio_path = upload_file.name
        # Copier sur Drive
        dest = os.path.join(F03_IN, os.path.basename(audio_path))
        shutil.copy(audio_path, dest)
    else:
        return None, None, None, '[ERREUR] Aucune entrée.'

    status.append(f'[F03A] Analyse : {os.path.basename(audio_path)}')
    t0 = time.time()

    try:
        result = analyze_audio(audio_path)
        _last_analysis = result
        _last_analysis['audio_path'] = audio_path

        png = plot_analysis(
            result['y'], result['sr'],
            result['bpm'], result['beat_times'],
            result['bounds_times'], audio_path
        )
        df = segments_to_df(result['segments'])

        status.append(f'[F03A] BPM         : {result["bpm"]}')
        status.append(f'[F03A] Durée       : {result["duration"]}s')
        status.append(f'[F03A] Sections    : {len(result["segments"])}')
        status.append(f'[F03A] Terminé en  : {round(time.time()-t0, 2)}s')

        return df, png, result['bpm'], '\n'.join(status)

    except Exception as e:
        import traceback
        return None, None, None, f'[ERREUR] {e}\n{traceback.format_exc()}'


def export_directives(df_data, bpm_val, crossfade_ms, ducking_db):
    """Convertir la DataFrame en directives.json et sauvegarder."""
    if df_data is None or (hasattr(df_data, 'empty') and df_data.empty):
        return '[ERREUR] Aucun segment. Lancer l\'analyse d\'abord.'

    if not isinstance(df_data, pd.DataFrame):
        df_data = pd.DataFrame(df_data)

    segments = []
    for _, row in df_data.iterrows():
        segments.append({
            'role'       : str(row.get('role', 'loop')),
            'start'      : float(row.get('start', 0)),
            'end'        : float(row.get('end', 0)),
            'loops'      : int(row.get('loops', 1)),
            'speed'      : float(row.get('speed', 1.0)),
            'reverse'    : bool(row.get('reverse', False)),
            'volume_pct' : int(row.get('volume_pct', 100)),
            'fade_in_ms' : int(row.get('fade_in_ms', 0)),
            'fade_out_ms': int(row.get('fade_out_ms', 15)),
        })

    audio_path = _last_analysis.get('audio_path', '')
    directives = {
        'source_music'  : os.path.basename(audio_path),
        'bpm'           : float(bpm_val) if bpm_val else _last_analysis.get('bpm', 128.0),
        'total_duration_sec': _last_analysis.get('duration', 0.0),
        'audio_timeline': segments,
        'crossfade_ms'  : int(crossfade_ms),
        'ducking_db'    : float(ducking_db),
    }

    # Sauvegarder sur Drive
    out_drive = os.path.join(F03_CODEBASE, 'directives.json')
    with open(out_drive, 'w', encoding='utf-8') as f:
        json.dump(directives, f, indent=2, ensure_ascii=False)

    # Sauvegarder aussi dans le repo local (pour SR_CUSTOS)
    os.makedirs(LOCAL_CODEBASE, exist_ok=True)
    with open(os.path.join(LOCAL_CODEBASE, 'directives.json'), 'w') as f:
        json.dump(directives, f, indent=2, ensure_ascii=False)

    # Mettre à jour liber
    if os.path.exists(LIBER_DRIVE):
        with open(LIBER_DRIVE) as f:
            liber = json.load(f)
        liber['bpm']                          = directives['bpm']
        liber['f03_seraphim']['status']        = 'directives_ready'
        liber['f03_seraphim']['directives_path'] = out_drive
        liber['f03_seraphim']['music_canvas']  = audio_path
        liber['fleet_status']                  = 'directives_ready'
        liber['sr_custos']['last_validation']  = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')
        with open(LIBER_DRIVE, 'w') as f:
            json.dump(liber, f, indent=2, ensure_ascii=False)

    # Log
    log_path = os.path.join(F03_TRACKING, 'F03_LOG.md')
    ts = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')
    with open(log_path, 'a') as f:
        f.write(f'\n## [{ts}] DIRECTIVES EXPORTÉES\n'
                f'bpm={directives["bpm"]} | segments={len(segments)} | '
                f'crossfade={crossfade_ms}ms | ducking={ducking_db}dB\n')

    return (f'[F03A] directives.json exporté → {out_drive}\n'
            f'[F03A] BPM: {directives["bpm"]} | {len(segments)} segments\n'
            f'[F03A] fleet_status → directives_ready\n'
            f'[F03A] Prêt pour F03B — La Machine à Micro-jets')


def add_segment(df_data):
    if df_data is None or (hasattr(df_data, 'empty') and df_data.empty):
        df_data = pd.DataFrame(columns=SEG_COLUMNS)
    elif not isinstance(df_data, pd.DataFrame):
        df_data = pd.DataFrame(df_data)
    new_row = pd.DataFrame([{
        'role': 'loop', 'start': 0.0, 'end': 10.0, 'loops': 1,
        'speed': 1.0, 'reverse': False, 'volume_pct': 100,
        'fade_in_ms': 15, 'fade_out_ms': 15
    }])
    return pd.concat([df_data, new_row], ignore_index=True)


def remove_last_segment(df_data):
    if df_data is None or (hasattr(df_data, 'empty') and df_data.empty):
        return df_data
    if not isinstance(df_data, pd.DataFrame):
        df_data = pd.DataFrame(df_data)
    return df_data.iloc[:-1] if len(df_data) > 0 else df_data


# ╔══════════════════════════════════════════════════════════╗
# ║  INTERFACE GRADIO                                        ║
# ╚══════════════════════════════════════════════════════════╝

CSS = """
.gradio-container { background: #08080e; }
#seraphim-a-header { text-align:center; padding:16px; border:1px solid #2a1a3a;
    background: linear-gradient(135deg, #100520 0%, #08080e 100%); margin-bottom:12px; }
#seraphim-a-header h1 { color: #9b59b6; font-family:monospace; font-size:1.4em; }
#seraphim-a-header p  { color:#666; font-size:0.85em; font-family:monospace; }
button.primary { background: #2a1a3a !important; color: #d7b8f5 !important; }
"""

with gr.Blocks(css=CSS, title='F03A SERAPHIM-A — L\'Architecte') as demo:

    gr.HTML("""
    <div id='seraphim-a-header'>
      <h1>F03A — SERAPHIM-A : L'Architecte</h1>
      <p>Analyse BPM · Détection sections · Éditeur directives JSON</p>
      <p>Ordre de la Rose Sacrée — Adepta Sororitas</p>
    </div>
    """)

    with gr.Tabs():

        # ── Onglet 1 : Analyse ─────────────────────────────────────────
        with gr.Tab('Analyse Audio'):
            with gr.Row():
                with gr.Column(scale=2):
                    use_drive = gr.Checkbox(
                        label='Utiliser la musique depuis Drive (F03_SERAPHIM/IN/)',
                        value=True
                    )
                    upload_music = gr.File(
                        label='Ou uploader un fichier audio (.mp3 / .wav / .flac)',
                        file_types=['.mp3', '.wav', '.flac', '.ogg', '.m4a']
                    )
                    analyze_btn = gr.Button('ANALYSER — SERAPHIM-A', variant='primary', size='lg')
                    bpm_out     = gr.Number(label='BPM détecté', precision=2)
                    status_out  = gr.Textbox(label='Rapport', lines=6, interactive=False)

                with gr.Column(scale=3):
                    analysis_img = gr.Image(
                        label='Waveform + BPM grid + Sections',
                        type='filepath'
                    )

        # ── Onglet 2 : Éditeur Timeline ────────────────────────────────
        with gr.Tab('Éditeur Timeline'):
            gr.Markdown("""
            ### Éditer la timeline — chaque ligne = un segment
            **Rôles** : `queue` (intro) · `loop` (refrain) · `tete` (drop) · `drop` · `bridge` · `outro`  
            **speed** : 1.0 = normal · >1 = rapide · <1 = lent (pitch préservé via pyrubberband)
            """)
            segments_df = gr.DataFrame(
                headers=SEG_COLUMNS,
                datatype=['str','number','number','number','number','bool','number','number','number'],
                label='Timeline — segments',
                interactive=True,
                row_count=(5, 'dynamic'),
                col_count=(9, 'fixed'),
            )
            with gr.Row():
                add_btn    = gr.Button('+ Ajouter segment', size='sm')
                remove_btn = gr.Button('- Supprimer dernier', size='sm')

            gr.Markdown('### Paramètres globaux')
            with gr.Row():
                bpm_override = gr.Number(label='BPM (override)', value=128.0, precision=2)
                crossfade_ms = gr.Slider(
                    label='Crossfade entre segments (ms)', minimum=0, maximum=200, step=5, value=15
                )
                ducking_db = gr.Slider(
                    label='Ducking musique sous voix (dB)',
                    minimum=-24, maximum=0, step=1, value=-14
                )

            export_btn    = gr.Button('EXPORTER directives.json → Drive', variant='primary', size='lg')
            export_status = gr.Textbox(label='Statut export', lines=5, interactive=False)

        # ── Onglet 3 : Prévisualisation JSON ───────────────────────────
        with gr.Tab('Prévisualisation JSON'):
            gr.Markdown('### directives.json — aperçu avant export')
            json_preview = gr.JSON(label='directives.json')
            preview_btn  = gr.Button('Générer prévisualisation')

        # ── Onglet 4 : Statut Flotte ───────────────────────────────────
        with gr.Tab('Statut Flotte'):
            liber_out   = gr.JSON(label='liber_sanctorum.json')
            refresh_btn = gr.Button('Lire le Liber')

    # ── Câblage ──────────────────────────────────────────────────────────

    analyze_btn.click(
        run_analysis,
        inputs=[use_drive, upload_music],
        outputs=[segments_df, analysis_img, bpm_out, status_out]
    )

    add_btn.click(add_segment, inputs=[segments_df], outputs=[segments_df])
    remove_btn.click(remove_last_segment, inputs=[segments_df], outputs=[segments_df])

    export_btn.click(
        export_directives,
        inputs=[segments_df, bpm_override, crossfade_ms, ducking_db],
        outputs=[export_status]
    )

    preview_btn.click(
        lambda df, bpm, cf, dk: {
            'bpm': bpm, 'crossfade_ms': cf, 'ducking_db': dk,
            'segments_count': len(df) if df is not None else 0,
            'segments': df.to_dict('records') if isinstance(df, pd.DataFrame) else []
        },
        inputs=[segments_df, bpm_override, crossfade_ms, ducking_db],
        outputs=[json_preview]
    )

    refresh_btn.click(
        lambda: json.load(open(LIBER_DRIVE)) if os.path.exists(LIBER_DRIVE) else {},
        inputs=[],
        outputs=[liber_out]
    )

print('[GRADIO] Démarrage interface L\'Architecte...')
demo.launch(share=True, debug=False, server_port=7862, inbrowser=False)

---
## CELLULE 4 — SR_CUSTOS CHECK-IN F03A

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELLULE 4 — SR_CUSTOS CHECK-IN F03A                    ║
# ║  Valide directives.json + enregistre dans la flotte      ║
# ╚══════════════════════════════════════════════════════════╝

import os, sys, json, shutil

DRIVE_ROOT    = '/content/drive/MyDrive/SANCTORUM'
SANCTORUM_DIR = '/content/SANCTORUM'
LIBER_DRIVE   = f'{DRIVE_ROOT}/liber_sanctorum.json'
OUT_PATH      = f'{DRIVE_ROOT}/F03_SERAPHIM/CODEBASE/directives.json'

if not os.path.exists(OUT_PATH):
    print('[SR_CUSTOS] ERREUR: directives.json absent. Lancer la cellule 3 d\'abord.')
    print('            → Analyser une musique ET cliquer "EXPORTER directives.json".')
else:
    custos = os.path.join(SANCTORUM_DIR, 'SR_CUSTOS.py')
    if os.path.exists(custos):
        local_liber = os.path.join(SANCTORUM_DIR, 'liber_sanctorum.json')
        shutil.copy(LIBER_DRIVE, local_liber)
        !python {custos} --mode check-in --frigate F03A --output {OUT_PATH}
        shutil.copy(local_liber, LIBER_DRIVE)
    else:
        print('[SR_CUSTOS] SR_CUSTOS.py absent — mise à jour manuelle.')
        with open(LIBER_DRIVE) as f:
            liber = json.load(f)
        liber['f03_seraphim']['status']          = 'directives_ready'
        liber['f03_seraphim']['directives_path'] = OUT_PATH
        liber['fleet_status']                    = 'directives_ready'
        liber['sr_custos']['last_validation']    = __import__('datetime').datetime.now(
            __import__('datetime').timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')
        with open(LIBER_DRIVE, 'w') as f:
            json.dump(liber, f, indent=2, ensure_ascii=False)

    with open(LIBER_DRIVE) as f:
        liber = json.load(f)
    print('\n╔══════════════════════════════════════════════════════╗')
    print('║          ÉTAT DE LA FLOTTE — POST F03A               ║')
    print('╠══════════════════════════════════════════════════════╣')
    print(f"║  fleet_status : {liber.get('fleet_status','n/a'):<36}║")
    print(f"║  F01 DOMINION : {liber['f01_dominion']['status']:<36}║")
    print(f"║  F02 CELESTIAN: {liber['f02_celestian']['status']:<36}║")
    print(f"║  F03 SERAPHIM : {liber['f03_seraphim']['status']:<36}║")
    print('╠══════════════════════════════════════════════════════╣')
    print('║  PROCHAINE ÉTAPE : F03B — La Machine à Micro-jets   ║')
    print('╚══════════════════════════════════════════════════════╝')
